# Lattice Methods for Derivative Pricing

From-scratch implementations of trinomial trees, exotic options on lattices, and interest rate trees.

**Outline**
1. Motivation -- trees as discrete approximations
2. Binomial tree recap
3. Trinomial tree -- why add a middle branch
4. Trinomial tree visualization
5. Barrier options on trees
6. Adaptive mesh refinement near barriers
7. Binomial vs trinomial convergence
8. Interest rate trees -- from equity to fixed income
9. Hull-White trinomial tree
10. Bond options on lattice
11. Summary
12. References

---

## Why Lattice Methods?

Lattice methods are among the most intuitive and flexible tools in computational finance. The core idea is beautifully simple:

> **Key Concept:** A lattice (or tree) approximates the continuous evolution of an asset price by breaking time into discrete steps. At each step, the price can move to a small number of possible values. By working backwards from the option's expiry payoff, we can determine today's fair price.

Think of it like planning a road trip. Instead of tracking your exact GPS position every millisecond, you note which city you're in at the end of each hour. The more "checkpoints" (time steps) you use, the closer your discrete route matches the actual continuous journey.

### Trees as Discrete Approximations to Continuous Stock Price Evolution

In the real world, stock prices move in continuous time. The standard mathematical model is **Geometric Brownian Motion (GBM)**:

$$dS = \mu S \, dt + \sigma S \, dW$$

This stochastic differential equation says that at every infinitesimal instant, the stock price experiences a deterministic drift ($\mu S \, dt$) and a random shock ($\sigma S \, dW$, where $dW$ is a Wiener process increment). The solution gives us log-normally distributed stock prices.

A **lattice** (or **tree**) replaces this continuous process with a discrete one. Instead of the stock price moving continuously through every possible value, we:

1. **Divide time into $N$ discrete steps**, each of size $\Delta t = T/N$
2. **At each step, allow the stock to jump to a small number of values** (2 for binomial, 3 for trinomial)
3. **Choose the jump sizes and probabilities** so that the discrete process has the same mean and variance as the continuous one

As $N \to \infty$ (equivalently, $\Delta t \to 0$), the discrete tree converges to the continuous GBM. This is not just a numerical trick -- it is a deep mathematical fact rooted in the Central Limit Theorem. After $N$ steps, each contributing a small random increment, the sum of those increments approaches a normal distribution, which is exactly what GBM produces in the log-price.

**Why is this useful?** Because the discrete tree gives us a concrete, computable object. At every node, we can:
- Check whether to exercise an American option
- Check whether a barrier has been breached
- Apply path-dependent payoff conditions
- Read off the local interest rate (for rate trees)

None of these things are easy in continuous time, but they are straightforward on a tree.

### What You Will Learn

| Topic | Key Question It Answers |
|:------|:-----------------------|
| Binomial trees | How does the simplest discrete model price options? |
| Trinomial trees | Why might three branches be better than two? |
| Barrier options | How do we handle knock-in/knock-out features on a tree? |
| Adaptive mesh | Can we refine the grid where it matters most? |
| Interest rate trees | How do lattice methods extend to fixed-income derivatives? |
| Hull-White model | How do we build a tree that matches today's yield curve? |

### Prerequisites

You should be comfortable with:
- Basic option payoffs (calls, puts, European vs American)
- The concept of risk-neutral pricing (discounting expected payoffs at the risk-free rate)
- The binomial model at an introductory level (we'll review it here)

If any of these are unfamiliar, don't worry -- we provide a thorough review of each concept before we use it.


### Why Go Beyond Binomial?

The binomial tree works, but it has some drawbacks:

1. **Slow convergence:** Prices oscillate (even vs odd steps) as $N$ increases
2. **Limited flexibility:** Only two branches constrain the ability to match higher moments
3. **Barrier alignment:** Barrier levels rarely fall exactly on tree nodes, causing pricing errors

The **trinomial tree** addresses these issues by allowing three moves at each step: up, middle, and down.

> **Analogy:** A binomial tree is like giving driving directions with only "turn left" or "turn right." A trinomial tree adds "go straight" -- you can describe the route more accurately with fewer intersections.

### The Trinomial Parameterisation

At each node, the stock can move to one of three values:

$$S \times u, \quad S \times m, \quad S \times d$$

with probabilities $p_u$, $p_m$, and $p_d$ respectively, where $p_u + p_m + p_d = 1$.

A common choice (matching the first two moments of GBM):

$$u = e^{\sigma\sqrt{2\Delta t}}, \quad d = e^{-\sigma\sqrt{2\Delta t}}, \quad m = 1$$

$$p_u = \left(\frac{e^{r\Delta t/2} - e^{-\sigma\sqrt{\Delta t/2}}}{e^{\sigma\sqrt{\Delta t/2}} - e^{-\sigma\sqrt{\Delta t/2}}}\right)^2$$

$$p_d = \left(\frac{e^{\sigma\sqrt{\Delta t/2}} - e^{r\Delta t/2}}{e^{\sigma\sqrt{\Delta t/2}} - e^{-\sigma\sqrt{\Delta t/2}}}\right)^2$$

$$p_m = 1 - p_u - p_d$$

> **Key Concept:** The extra "middle" branch means the trinomial tree is equivalent to two steps of a binomial tree combined into one. This is why it converges faster -- it effectively uses twice as many time steps for the same computational grid.

**Comparison: Binomial vs Trinomial**

| Feature | Binomial | Trinomial |
|:--------|:---------|:----------|
| Branches per node | 2 | 3 |
| Convergence speed | $O(1/N)$ | $O(1/N^2)$ |
| Oscillation | Even/odd pattern | Smooth |
| Nodes at step $n$ | $n + 1$ | $2n + 1$ |
| Barrier alignment | Difficult | More flexible |
| Implementation | Simpler | Slightly more complex |

Let's implement the trinomial tree and compare.


### Why Are Barrier Options Tricky on Trees?

Barrier options have a payoff that depends on whether the underlying price **touches a certain level** (the barrier) during the life of the option. Types include:

| Type | Meaning |
|:-----|:--------|
| **Up-and-out** | Option dies if price rises above barrier |
| **Down-and-out** | Option dies if price falls below barrier |
| **Up-and-in** | Option activates only if price rises above barrier |
| **Down-and-in** | Option activates only if price falls below barrier |

> **Real-world example:** A down-and-out call on a stock at $100 with barrier at $80 and strike at $95. This gives you a cheaper call (because it can "knock out"), but if the stock ever drops to $80, your option vanishes -- worthless.

**What Do Knock-In and Knock-Out Mean?**

These terms describe how the barrier affects the option's existence:

- **Knock-out** options start alive and **die** if the barrier is hit. You hold a regular option, but it has a self-destruct mechanism. If the stock price touches the barrier at any point before expiry, the option immediately becomes worthless -- regardless of where the stock ends up at maturity. You might hear traders say the option "knocks" or gets "knocked."

- **Knock-in** options start dead and **activate** if the barrier is hit. You hold a contract that only becomes a real option once the stock price touches the barrier. Until that barrier-hit event occurs, the contract pays nothing at expiry, even if the final stock price would have produced a large payoff.

**Why would anyone want these?** Cost. Because a knock-out call can be extinguished, it is always cheaper than a vanilla call. A corporate treasurer hedging currency risk might buy a knock-out option at a lower premium, accepting the risk that extreme moves could eliminate the hedge. Conversely, a speculator who believes a stock will first dip before rallying might prefer a knock-in put -- paying less premium for a bet on a specific price path.

**The Node Alignment Problem:**

On a discrete tree, the barrier level almost never falls exactly on a node. If the barrier is at $80 but the nearest tree nodes are at $78.50 and $81.20, what do we do?

- If we knock out at $81.20$ (the nearest node above), we knock out too often -- price too low
- If we knock out at $78.50$ (nearest node below), we knock out too rarely -- price too high

This **specification error** can be significant, especially for short-dated or near-the-money barriers.

> **Key Concept:** The trinomial tree helps because it has more nodes (3 branches vs 2), making it more likely that nodes align closely with the barrier. Adaptive mesh refinement goes further by placing nodes exactly on the barrier.

> **Common Mistake:** A frequent beginner error is to check the barrier condition only at the terminal nodes. The barrier must be monitored at **every** node, at **every** time step. If the stock breaches the barrier at any intermediate step, the knock-out option dies immediately -- you cannot "undo" the breach just because the stock later recovers.

**Barrier Parity:**

A useful relationship for checking your work:

$$\text{Knock-in} + \text{Knock-out} = \text{Vanilla}$$

This must hold for European options. If your knock-in + knock-out prices don't sum to the vanilla price, something is wrong.

> **Important:** Barrier parity holds exactly in continuous time but only approximately on a discrete tree (due to the node alignment problem). The discrepancy shrinks as $N$ increases. Always use this identity as a sanity check -- if the sum is wildly different from the vanilla price, you have a bug.

Let's implement barrier option pricing on the trinomial tree.


### Why Do We Need Special Trees for Interest Rates?

For equities, the stock price today is observable and we build the tree forward from there. For interest rates, the situation is more complex:

1. **The yield curve already exists.** Unlike a single stock price, we have a whole term structure of rates. Our tree must reproduce the current market yield curve -- otherwise, it would misprice existing bonds.

2. **Interest rates mean-revert.** Unlike stocks, rates don't wander off to infinity. The Hull-White model captures this:

$$dr = [\theta(t) - ar]\,dt + \sigma\,dW$$

where:
- $a$ = speed of mean reversion ("how quickly rates are pulled back to normal")
- $\sigma$ = short-rate volatility
- $\theta(t)$ = time-dependent drift, calibrated to match the current yield curve

> **Intuition:** Think of $a$ as the strength of a rubber band pulling the rate back to its equilibrium. The function $\theta(t)$ shifts that equilibrium over time so that the model perfectly reproduces today's bond prices.

**Building the Tree:**

The Hull-White trinomial tree is built in two stages:

1. **Stage 1:** Build a tree for a "symmetrised" process centered at zero, using the mean-reversion and volatility parameters
2. **Stage 2:** Shift every node by $\alpha(t)$ -- a time-dependent displacement that ensures the tree matches the current term structure

> **Why this matters:** A bond option priced on this tree will be consistent with today's yield curve. If you used a tree that didn't match the curve, you'd get an option price that implies the market is mispricing bonds -- which is almost certainly wrong.

Let's build a Hull-White trinomial tree.> **CFA Exam Tip:** Barrier options are classified as "exotic" options. They are cheaper than vanilla options because the barrier removes some payoff scenarios. The knock-out feature means the option can cease to exist, reducing its expected payoff and therefore its price. The knock-in feature means the option only activates if the barrier is hit, also reducing the expected payoff relative to a vanilla option.


---
## 1. Motivation -- Trees as Discrete Approximations

Lattice (tree) methods provide flexible, intuitive frameworks for pricing derivatives. The core idea:

**Approximate a continuous random process with a discrete one.**

Instead of the stock price moving continuously, it jumps to one of a few possible values at each time step. As we increase the number of steps, the discrete tree converges to the continuous process.

### Real-world analogy

Think of a tree as a "choose your own adventure" book for stock prices. At each moment, the stock can go up, down, or (in a trinomial tree) stay roughly the same. By the end of the book, there are thousands of possible endings (final stock prices). The option value is the average of the payoffs across all endings, weighted by their probabilities.

> **Key Concept:** Trees work by backward induction -- start at expiry (where we know the payoff), then work backward asking "what is this option worth one step earlier?" at each node. This naturally handles American exercise: at each node, we compare the continuation value with the exercise value.

### The Connection to Continuous Finance

Let us be precise about what "discrete approximation" means. Consider the stock price under GBM over a small interval $\Delta t$:

$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) \sim \mathcal{N}\left(\left(r - \frac{\sigma^2}{2}\right)\Delta t, \; \sigma^2 \Delta t\right)$$

The log-return is normally distributed with a specific mean and variance. When we build a tree, we choose the jump sizes ($u$, $d$, and possibly $m$) and the jump probabilities ($p_u$, $p_d$, and possibly $p_m$) so that the **discrete** distribution of log-returns has exactly the same first and second moments as this normal distribution.

- **First moment (mean):** ensures the tree grows at the risk-free rate (risk-neutral drift)
- **Second moment (variance):** ensures the tree captures the correct level of uncertainty (volatility)

With two branches (binomial), matching two moments uses up all our degrees of freedom. With three branches (trinomial), we have an extra free parameter, which we can use for additional flexibility (e.g., placing a node exactly on a barrier).

### Why trinomial trees?

While binomial trees (2 branches) are foundational, **trinomial trees** (3 branches) offer:
- **Better convergence** -- more accurate for the same number of steps
- **More flexibility** -- the middle branch provides an extra degree of freedom for matching moments
- **Natural for interest rate models** -- the Hull-White model is built on trinomial trees

### A Preview of Backward Induction

The pricing algorithm on any tree works in three phases:

1. **Build the tree forward:** Start at $t = 0$ with the current stock price. At each time step, compute all reachable stock prices using the up/down/middle factors.

2. **Set terminal payoffs:** At the final time step $t = T$ (expiry), compute the option payoff at every node. For a call, this is $\max(S_T - K, 0)$. For a put, it is $\max(K - S_T, 0)$.

3. **Work backward to today:** At each earlier time step, the option value at a node equals the discounted expected value of the option at the next time step's connected nodes. In a trinomial tree:

$$V_\text{node} = e^{-r\Delta t} \left[ p_u \cdot V_\text{up} + p_m \cdot V_\text{mid} + p_d \cdot V_\text{down} \right]$$

For American options, at each node we also check: is it better to exercise now or to continue holding? We take the maximum of the two:

$$V_\text{node} = \max\left(\text{exercise value}, \; e^{-r\Delta t} \left[ p_u V_\text{up} + p_m V_\text{mid} + p_d V_\text{down} \right]\right)$$

> **Important:** Backward induction is the engine that makes lattice methods work. Every pricing calculation in this notebook -- vanilla options, barrier options, bond options -- uses this same backward-stepping logic. Understand it once, and you understand all lattice pricing.### When to Use Trees vs Other Methods

| Method | Best for | Limitation |
|:-------|:---------|:-----------|
| **Analytical (BSM)** | European vanilla | Very few closed forms exist |
| **Binomial/trinomial tree** | American options, 1–2 underlyings, need early exercise boundary | Curse of dimensionality beyond ~3 assets |
| **Finite differences** | 1D–2D PDEs, smooth payoffs | Grid-based, similar scaling issues |
| **Monte Carlo** | Path-dependent, multi-asset, complex payoffs | Difficult for American options |

> **Key Concept:** Trees are uniquely well-suited for **American options** because backward induction naturally handles the early exercise decision at every node. Monte Carlo struggles with American options because it simulates *forward* in time — you don't know the continuation value without extra machinery (like Longstaff-Schwartz regression).

### The Recombining Property

A tree is **recombining** if an up move followed by a down move leads to the same node as a down move followed by an up. This keeps the number of nodes manageable:
- Recombining binomial: $N + 1$ nodes at step $N$ → total $O(N^2)$ nodes
- Non-recombining binomial: $2^N$ nodes at step $N$ → exponentially many

The CRR parameterisation ($u = 1/d$) ensures recombination.


---

### 2.1 The Binomial Model: A Quick Review

Before extending to trinomial trees, let's make sure the binomial foundation is solid.

**The Setup:** Consider a stock at price $S_0$. Over one time step $\Delta t$, it can either:
- Go **up** to $S_0 \times u$ with risk-neutral probability $p$
- Go **down** to $S_0 \times d$ with risk-neutral probability $1 - p$

The **Cox-Ross-Rubinstein (CRR)** parameterisation ensures the tree converges to geometric Brownian motion:

$$u = e^{\sigma\sqrt{\Delta t}}, \quad d = e^{-\sigma\sqrt{\Delta t}} = \frac{1}{u}$$

$$p = \frac{e^{r\Delta t} - d}{u - d}$$

where $\sigma$ is volatility and $r$ is the risk-free rate.

> **Intuition:** The up and down factors are chosen so that the tree's variance matches the stock's real volatility. The probability $p$ is chosen so that the expected growth rate equals the risk-free rate (risk-neutral pricing).

### Understanding Risk-Neutral Probability

The probability $p$ deserves special attention because it is one of the most misunderstood concepts in quantitative finance.

In the real world, if a stock has an expected return of 10% per year, the probability of going up is higher than the probability of going down. But we do **not** use this real-world probability for pricing. Instead, we use the **risk-neutral probability** $p$, which is constructed so that:

$$E^Q[S_{t+\Delta t}] = S_t \cdot e^{r \Delta t}$$

Under the risk-neutral measure $Q$, the stock grows at the risk-free rate $r$, not at its real expected return $\mu$.

> **Key Concept:** Risk-neutral pricing works "as if investors don't care about risk." In this fictional world, all assets earn the risk-free rate on average, and we can price derivatives by simply discounting expected payoffs at the risk-free rate. The mathematical justification comes from the Fundamental Theorem of Asset Pricing: the absence of arbitrage implies the existence of such a measure.

**Why does this work?** The key insight is that the option price does not depend on how risk-averse investors are, nor on the stock's real expected return. It depends only on the current stock price, the volatility, the risk-free rate, and the option's contractual terms. The risk-neutral probability is simply the unique probability that makes the discounted stock price a martingale (a fair game), which in turn ensures no-arbitrage pricing.

> **Common Mistake:** Beginners often try to estimate the "real" probability of an up-move (e.g., from historical data) and use it for pricing. This is incorrect. The risk-neutral probability is not a statistical estimate -- it is derived from the no-arbitrage condition. Using real-world probabilities would give you an expected payoff, not a fair price.

### Worked Example: 1-Period Binomial Tree

Suppose $S_0 = 100$, $\sigma = 0.20$, $r = 0.05$, $\Delta t = 1$ year.

**Step 1: Compute the up and down factors.**

$$u = e^{0.20 \times 1} = 1.2214, \quad d = \frac{1}{1.2214} = 0.8187$$

**Step 2: Compute the possible stock prices at $t = 1$.**

$$S_u = 100 \times 1.2214 = 122.14, \quad S_d = 100 \times 0.8187 = 81.87$$

**Step 3: Compute the risk-neutral probability.**

$$p = \frac{e^{0.05} - 0.8187}{1.2214 - 0.8187} = \frac{1.0513 - 0.8187}{0.4027} = 0.5775$$

**Step 4: Compute the call payoffs at expiry** (strike $K = 100$).

- If stock goes up: $\max(122.14 - 100, 0) = 22.14$
- If stock goes down: $\max(81.87 - 100, 0) = 0$

**Step 5: Discount the expected payoff to today.**

$$C_0 = e^{-0.05 \times 1}(0.5775 \times 22.14 + 0.4225 \times 0) = e^{-0.05} \times 12.79 = 12.16$$

The 1-period call price is approximately $12.16.

### Worked Example: 2-Period Binomial Tree

Now let us use $N = 2$ steps, so $\Delta t = 0.5$ years.

**Step 1: Recompute parameters for the finer time step.**

$$u = e^{0.20\sqrt{0.5}} = e^{0.1414} = 1.1519, \quad d = 1/u = 0.8681$$

$$p = \frac{e^{0.05 \times 0.5} - 0.8681}{1.1519 - 0.8681} = \frac{1.0253 - 0.8681}{0.2838} = 0.5539$$

**Step 2: Build the stock price tree forward.**

```
                      S_uu = 100 * 1.1519^2 = 132.69
                     /
        S_u = 115.19
       /             \
S0 = 100              S_ud = 100 * 1.1519 * 0.8681 = 100.00
       \             /
        S_d = 86.81
                     \
                      S_dd = 100 * 0.8681^2 = 75.36
```

Notice that $S_{ud} = S_{du} = S_0$ (the tree is **recombining**). This is a consequence of $u \times d = 1$.

**Step 3: Compute terminal payoffs** for a call with $K = 100$.

- $V_{uu} = \max(132.69 - 100, 0) = 32.69$
- $V_{ud} = \max(100.00 - 100, 0) = 0.00$
- $V_{dd} = \max(75.36 - 100, 0) = 0.00$

**Step 4: Backward induction -- step back from $t=1$ to $t=0.5$.**

At the up-node ($S_u = 115.19$):
$$V_u = e^{-0.05 \times 0.5}(0.5539 \times 32.69 + 0.4461 \times 0.00) = 0.9753 \times 18.11 = 17.66$$

At the down-node ($S_d = 86.81$):
$$V_d = e^{-0.05 \times 0.5}(0.5539 \times 0.00 + 0.4461 \times 0.00) = 0$$

**Step 5: Step back from $t=0.5$ to $t=0$.**

$$V_0 = e^{-0.05 \times 0.5}(0.5539 \times 17.66 + 0.4461 \times 0) = 0.9753 \times 9.78 = 9.54$$

The 2-period call price is approximately $9.54. Notice this differs from the 1-period price ($12.16) -- with only 1 or 2 steps, the tree is a coarse approximation. As $N$ increases, the price converges to the Black-Scholes value of $10.45.

> **Important:** The 2-period example illustrates the essence of backward induction: we never need to enumerate all paths. At each node, we simply combine the values from the next time step. This makes the algorithm $O(N^2)$ (for the total number of nodes), not $O(2^N)$ (for the total number of paths).

### Backward Induction: Step-by-Step Walkthrough

Let us formalize the backward induction algorithm so it is crystal clear.

**Given:** A recombining tree with $N$ time steps, stock prices $S(n, j)$ at step $n$ and node $j$, and an option with payoff function $\phi(S)$.

1. **Initialize terminal values.** At step $N$, for each node $j$:
   $$V(N, j) = \phi(S(N, j))$$
   For a European call: $\phi(S) = \max(S - K, 0)$. For a put: $\phi(S) = \max(K - S, 0)$.

2. **Step backward.** For $n = N-1, N-2, \ldots, 0$, and for each node $j$ at step $n$:
   $$V(n, j) = e^{-r\Delta t} \sum_{k} p_k \cdot V(n+1, j+k)$$
   where the sum is over the branches (for binomial: $k \in \{+1, -1\}$; for trinomial: $k \in \{+1, 0, -1\}$).

3. **American exercise check** (if applicable). At each node, compare continuation value with exercise value:
   $$V(n, j) = \max\left(V(n, j), \; \phi(S(n, j))\right)$$

4. **Read off the price.** The option's fair value today is $V(0, 0)$ -- the value at the root node.

> **Key Concept:** Backward induction transforms an exponentially complex problem (evaluating all possible paths) into a polynomial one (evaluating each node once). This is the same principle as dynamic programming in computer science.

> **CFA Exam Tip:** The risk-neutral probability $p$ is NOT the real-world probability that the stock goes up. It is a mathematical construction that allows us to price by discounting at the risk-free rate. The real-world probability is irrelevant for pricing.

Let's implement this and verify our hand calculation.

## 2. Setup

We begin by importing the necessary libraries and setting up global parameters. We use:

- **NumPy** for array operations and vectorized computation across tree nodes
- **SciPy** for the normal CDF (used in Black-Scholes) and optimization routines
- **Matplotlib** for visualizing trees, convergence plots, and price surfaces

We also define tolerance constants (`ATOL`, `RTOL`) that we will use later to verify our tree prices against known analytical solutions. If a tree price differs from the analytical benchmark by more than these tolerances, we know something is wrong.

> **Important:** All implementations in this notebook are built from scratch using NumPy. We deliberately avoid using pre-built option pricing libraries so that every step of the algorithm is transparent and editable.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

The cell above imports our core numerical libraries and configures the plotting style. The `SEED` ensures reproducible results if any randomness is used. The color palette (`PRIMARY`, `SECONDARY`, `TERTIARY`, `ACCENT`) keeps all plots visually consistent throughout the notebook.

Next, we implement the **Black-Scholes formula** for European calls and puts. This serves as our analytical benchmark -- the "correct" answer that our tree prices should converge to as $N \to \infty$. Having a reliable benchmark is essential: without it, we would have no way to know whether our tree implementations are correct.

Recall the Black-Scholes call formula:

$$C = S_0 \, \Phi(d_1) - K e^{-rT} \, \Phi(d_2)$$

where $\Phi$ is the standard normal CDF, and:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

The put formula follows from put-call parity: $P = K e^{-rT} \Phi(-d_2) - S_0 \Phi(-d_1)$.> **Note:** Throughout this notebook, we validate every tree implementation against analytical Black-Scholes prices. This is the gold standard for verification — if the tree price doesn't converge to BSM as $N$ increases, there's a bug. For products without analytical solutions (American options, barrier options), we use convergence checks and parity relations instead.


In [ ]:
# Black-Scholes for comparison
def bs_call(S, K, T, r, sigma):
    if T < 1e-14:
        return max(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    if T < 1e-14:
        return max(K - S, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)

We now have our Black-Scholes benchmark functions. Note the guard clause `if T < 1e-14` -- this handles the edge case where time-to-expiry is essentially zero, returning the intrinsic value directly and avoiding division by zero in the $d_1$ formula.

With the benchmark in place, let us build our first tree.> **Key Concept:** Having analytical benchmark prices is invaluable for debugging. When your tree gives a price of \$10.45 and BSM gives \$10.45, you know the implementation is correct. Without benchmarks, you'd have no way to distinguish a correct but slow-converging tree from a buggy one.

> **Key Concept:** The `bs_put` function uses **put-call parity** ($P = C - S + Ke^{-rT}$) rather than re-deriving the put formula. This is a common and elegant implementation technique — once you have the call price, the put comes for free via parity.


---
## 3. Binomial Tree Recap

The CRR (Cox-Ross-Rubinstein) tree:

$$u = e^{\sigma\sqrt{\Delta t}}, \quad d = 1/u, \quad p = \frac{e^{r\Delta t} - d}{u - d}$$

At each node: stock goes up by $u$ (prob $p$) or down by $d$ (prob $1-p$).

> **Key Concept:** The up/down factors are chosen to match the stock's volatility. The probability $p$ is chosen to match the risk-neutral drift. This ensures the tree converges to GBM as $\Delta t \to 0$.

### Implementation Details

The implementation below uses a **vectorized backward induction**. Rather than storing the entire tree (which would require $O(N^2)$ memory), we maintain only the current column of option values. At each backward step, we reduce the column by one element (binomial) or two elements (trinomial).

The terminal stock prices are computed using the formula:

$$S(N, j) = S_0 \cdot u^{N-j} \cdot d^{j}, \quad j = 0, 1, \ldots, N$$

where $j$ counts the number of down-moves. Since $d = 1/u$, this simplifies to $S(N, j) = S_0 \cdot u^{N - 2j}$.

For **American options**, at each backward step we compare the continuation value (discounted expected future value) with the immediate exercise value, and take the maximum. This is the key advantage of trees over the Black-Scholes formula: the formula can only price European options, while the tree handles early exercise naturally.

The code below implements all of this. After defining the function, we price a European call with $N = 200$ steps and compare to Black-Scholes.### Why Risk-Neutral Probabilities Work

The risk-neutral probability $p$ is NOT the real-world probability that the stock goes up. It is a mathematical construction:

> **Key Concept:** We choose $p$ so that the expected stock price grows at the risk-free rate $r$: $E[S_{t+1}] = S_t e^{r\Delta t}$. This means $p \cdot u + (1-p) \cdot d = e^{r\Delta t}$, giving $p = \frac{e^{r\Delta t} - d}{u - d}$.

Under these probabilities, the discounted stock price is a **martingale** — its expected future value (discounted) equals its current value. This is the fundamental theorem of asset pricing in discrete time.

### Hand-Worked 2-Period Example

$S_0 = 100$, $u = 1.1$, $d = 0.9$, $r = 2\%$ per period, European call with $K = 100$.

**Step 1 — Build the stock tree:**
```
        121  (uu)
    110
100     99   (ud = du)
     90
        81   (dd)
```

**Step 2 — Payoffs at expiry:** $C_{uu} = 21$, $C_{ud} = 0$, $C_{dd} = 0$

**Step 3 — Risk-neutral probability:** $p = \frac{1.02 - 0.9}{1.1 - 0.9} = 0.6$

**Step 4 — Backward induction:**
- $C_u = e^{-0.02}[0.6(21) + 0.4(0)] = e^{-0.02} \times 12.6 = 12.35$
- $C_d = e^{-0.02}[0.6(0) + 0.4(0)] = 0$
- $C_0 = e^{-0.02}[0.6(12.35) + 0.4(0)] = e^{-0.02} \times 7.41 = 7.26$


In [ ]:
def binomial_tree(S0, K, T, r, sigma, N, option_type='call', american=False):
    """CRR binomial tree option pricing."""
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    
    # Terminal stock prices
    S = S0 * u**(np.arange(N, -1, -1)) * d**(np.arange(0, N + 1, 1))
    
    # Terminal option values
    if option_type == 'call':
        V = np.maximum(S - K, 0)
    else:
        V = np.maximum(K - S, 0)
    
    # Backward induction
    for i in range(N - 1, -1, -1):
        S_nodes = S0 * u**(np.arange(i, -1, -1)) * d**(np.arange(0, i + 1, 1))
        V = disc * (p * V[:-1] + (1 - p) * V[1:])
        
        if american:
            if option_type == 'call':
                V = np.maximum(V, S_nodes - K)
            else:
                V = np.maximum(V, K - S_nodes)
    
    return V[0]

S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
bs_ref = bs_call(S0, K, T, r, sigma)
bt_val = binomial_tree(S0, K, T, r, sigma, 200, 'call')
print(f"Binomial tree (N=200): ${bt_val:.4f}")
print(f"Black-Scholes:         ${bs_ref:.4f}")

### Interpreting the Binomial Tree Result

With 200 time steps, the binomial tree price is very close to the Black-Scholes analytical value. The remaining difference (typically a few cents on a ~$10 option) comes from the discretization error -- the tree is still a finite approximation to the continuous process.

Let us verify the numbers match our earlier hand calculation. With the same parameters ($S_0 = 100$, $K = 100$, $T = 1$, $r = 5\%$, $\sigma = 20\%$), the Black-Scholes price should be approximately $10.45. Our 200-step binomial tree should agree to within a few basis points.

> **Common Mistake:** A frequent error when implementing binomial trees is to confuse the indexing convention. Some texts index from the top of the tree (highest stock price), others from the bottom. The backward induction formula `V = disc * (p * V[:-1] + (1 - p) * V[1:])` assumes that `V[:-1]` corresponds to the "up" nodes and `V[1:]` to the "down" nodes. Reversing this swaps $p$ and $1-p$, giving incorrect prices.

Now let us extend to three branches.> **Key Concept:** Backward induction is the heart of tree pricing. We start at the final nodes (where we know the payoff) and work backwards, discounting expected values at each step. For American options, we compare the "hold" value (discounted expected value) with the "exercise" value (intrinsic value) at each node, choosing the maximum.

> **CFA Exam Tip:** On the CFA exam, you may need to price a 2-period binomial option by hand. The process is:
> 1. Build the stock tree forward: $S_{uu} = S_0 u^2$, $S_{ud} = S_0 ud$, $S_{dd} = S_0 d^2$
> 2. Compute payoffs at expiry: $\max(S - K, 0)$ for calls
> 3. Work backwards: $C_{\text{node}} = e^{-r\Delta t}[p \cdot C_{\text{up}} + (1-p) \cdot C_{\text{down}}]$


---
## 4. Trinomial Tree -- Why Add a Middle Branch

A trinomial tree has three branches: up, middle, down.

$$u = e^{\sigma\sqrt{2\Delta t}}, \quad d = 1/u, \quad m = 1$$

### Why Three Branches?

The fundamental reason for the middle branch is that it provides an additional degree of freedom. In a binomial tree, we have three parameters to choose ($u$, $d$, and $p$), but we need to satisfy two constraints (matching the mean and variance of the stock's log-return), leaving one free parameter (typically fixed by setting $d = 1/u$). In a trinomial tree, we have five parameters ($u$, $d$, $m$, $p_u$, $p_d$, with $p_m = 1 - p_u - p_d$), and the same two constraints, leaving **three** free parameters.

The middle branch -- where the stock price stays roughly the same ($m = 1$) -- represents the scenario where **nothing dramatic happens** during the time step. In the real world, most of the time, stock prices don't make large moves. The trinomial tree captures this directly: the middle branch carries the most probability mass (typically $p_m \approx 2/3$), reflecting the fact that small moves are most common.

> **Key Concept:** The middle branch means "the stock stays roughly flat this period." In a binomial tree, this scenario is absent -- the stock must always move. The trinomial tree's ability to represent a flat move makes it a better match for the actual distribution of short-term returns, where the most likely outcome is a small change.

### The Trinomial-Binomial Equivalence

There is a beautiful connection: **one step of a trinomial tree is equivalent to two steps of a binomial tree**. Consider two successive binomial steps: up-then-down gives $S \cdot u \cdot d = S$ (if $ud = 1$), which is the same as the trinomial middle branch. Up-then-up gives $S \cdot u^2$, which matches the trinomial up branch (since the trinomial uses $u_{\text{tri}} = u_{\text{bin}}^2$). Down-then-down gives $S \cdot d^2$, matching the trinomial down branch.

This equivalence explains why the trinomial tree converges faster: each trinomial step does the work of two binomial steps.

**Why the middle branch helps:**
1. **Better accuracy:** Three branches match more moments of the distribution than two.
2. **More flexibility:** The extra branch lets us calibrate to more complex processes.
3. **Barrier alignment:** We can position nodes exactly at barrier levels (important for barrier options).

The probabilities match the first two moments of the stock's log-return:

$$p_u = \left(\frac{e^{r\Delta t/2} - e^{-\sigma\sqrt{\Delta t/2}}}{e^{\sigma\sqrt{\Delta t/2}} - e^{-\sigma\sqrt{\Delta t/2}}}\right)^2, \quad p_d = \left(\frac{e^{\sigma\sqrt{\Delta t/2}} - e^{r\Delta t/2}}{e^{\sigma\sqrt{\Delta t/2}} - e^{-\sigma\sqrt{\Delta t/2}}}\right)^2, \quad p_m = 1 - p_u - p_d$$

> **Important:** The trinomial tree is a **recombining** lattice. At step $n$, the number of distinct nodes is $2n + 1$ (not $3^n$, which would be the case for a non-recombining tree). An up-then-down move reaches the same node as a down-then-up move, or a middle-then-middle move. This recombination is what makes the algorithm computationally tractable.

### Node Structure

At step $n$, the stock prices on the trinomial tree are:

$$S(n, j) = S_0 \cdot u^j, \quad j = -n, -n+1, \ldots, n-1, n$$

So there are $2n + 1$ nodes at step $n$. The node index $j$ represents the net number of up-moves minus down-moves. The highest node ($j = n$) corresponds to all up-moves; the lowest ($j = -n$) to all down-moves; and $j = 0$ means the stock is back at $S_0$.

The following code implements the trinomial tree pricer. After defining it, we compare to both the binomial tree and Black-Scholes.### Trinomial vs Binomial: A Detailed Comparison

| Feature | Binomial (CRR) | Trinomial |
|:--------|:--------------|:----------|
| Branches per node | 2 | 3 |
| Nodes at step $N$ | $N + 1$ | $2N + 1$ |
| Total nodes | $O(N^2/2)$ | $O(N^2)$ |
| Convergence | $O(1/N)$ with oscillation | $O(1/N)$ smooth |
| Even-odd oscillation | Yes (significant) | Minimal |
| Equivalent FD scheme | Explicit | Implicit |
| Stability | Conditional | Unconditional |

> **Key Concept:** The trinomial tree's middle branch ($m = 1$, stock stays flat) captures the possibility that the stock price doesn't move much. This is important because in reality, small moves are the most common outcome. The binomial tree forces every move to be either "up" or "down," which overstates the actual volatility of the discrete steps.

> **Important:** Despite having ~2x as many nodes, the trinomial tree often requires FEWER steps for the same accuracy, making it computationally competitive with or better than the binomial tree.


In [ ]:
def trinomial_tree(S0, K, T, r, sigma, N, option_type='call', american=False):
    """Trinomial tree option pricing."""
    dt = T / N
    u = np.exp(sigma * np.sqrt(2 * dt))
    d = 1 / u
    m = 1.0
    
    # Probabilities
    sqrt_dt_half = np.sqrt(dt / 2)
    exp_up = np.exp(sigma * sqrt_dt_half)
    exp_dn = np.exp(-sigma * sqrt_dt_half)
    exp_r = np.exp(r * dt / 2)
    
    pu = ((exp_r - exp_dn) / (exp_up - exp_dn))**2
    pd = ((exp_up - exp_r) / (exp_up - exp_dn))**2
    pm = 1 - pu - pd
    
    disc = np.exp(-r * dt)
    
    # At step n, there are 2n+1 nodes
    # Node j at step n has price S0 * u^j where j ranges from -n to n
    
    # Terminal values (step N)
    j = np.arange(-N, N + 1)
    S_terminal = S0 * u**j
    
    if option_type == 'call':
        V = np.maximum(S_terminal - K, 0)
    else:
        V = np.maximum(K - S_terminal, 0)
    
    # Backward induction
    for n in range(N - 1, -1, -1):
        V_new = np.zeros(2 * n + 1)
        j_range = np.arange(-n, n + 1)
        S_nodes = S0 * u**j_range
        
        for idx in range(2 * n + 1):
            # Map to indices in V (which has 2*(n+1)+1 elements)
            # j at step n corresponds to j-1, j, j+1 at step n+1
            idx_in_V = idx  # V index at step n+1 for node j-1
            V_new[idx] = disc * (pu * V[idx + 2] + pm * V[idx + 1] + pd * V[idx])
        
        if american:
            if option_type == 'call':
                V_new = np.maximum(V_new, S_nodes - K)
            else:
                V_new = np.maximum(V_new, K - S_nodes)
        
        V = V_new
    
    return V[0]

tri_val = trinomial_tree(S0, K, T, r, sigma, 200, 'call')
print(f"Trinomial tree (N=200): ${tri_val:.4f}")
print(f"Black-Scholes:          ${bs_ref:.4f}")
print(f"Error:                  ${abs(tri_val - bs_ref):.6f}")

### Interpreting the Trinomial Tree Result

The trinomial tree with 200 steps produces a price very close to Black-Scholes. Compare the error with the binomial tree at the same number of steps -- the trinomial error is typically smaller, reflecting its faster convergence.

A few things to note about the implementation:

1. **The backward induction loop** iterates from `n = N-1` down to `n = 0`. At each step, the option value array shrinks from $2(n+1) + 1$ elements to $2n + 1$ elements.

2. **The index mapping** `V[idx + 2]`, `V[idx + 1]`, `V[idx]` connects each node at step $n$ to its three children at step $n + 1$. The up-child is at index `idx + 2`, the middle-child at `idx + 1`, and the down-child at `idx`.

3. **The American exercise check** is identical to the binomial case: compare continuation value with exercise value and take the maximum.

> **Common Mistake:** When implementing trinomial trees, a common indexing error is to confuse the node ordering. In our convention, node index 0 at step $n$ corresponds to $j = -n$ (the lowest stock price), and node index $2n$ corresponds to $j = +n$ (the highest). The "up" child of node `idx` is `idx + 2` (not `idx + 1`), because the tree adds one node above and one below at each step.

Now let us visualize the trinomial tree structure to build further intuition.

---
## 5. Trinomial Tree Visualization

Visualizing the tree makes the structure concrete. In the plot below, each dot represents a node (a possible stock price at a given time step). The lines show the three possible transitions from each node:

- **Green lines** = up moves (stock price increases)
- **Grey lines** = middle moves (stock price stays roughly the same)
- **Coral lines** = down moves (stock price decreases)

Notice how the tree fans out as time progresses. At step 0, there is one node ($S_0$). At step 1, there are 3 nodes. At step 2, there are 5 nodes. At step $n$, there are $2n + 1$ nodes.

Also notice that the tree is **symmetric on a log scale** -- the spacing between nodes appears to increase as prices get higher, which reflects the multiplicative nature of stock returns. The up factor $u$ and down factor $d = 1/u$ ensure that an up-move followed by a down-move returns to the original price ($S \cdot u \cdot d = S$).

> **Key Concept:** The recombining property ($u \times d = 1$) is what keeps the tree computationally tractable. Without recombination, the number of nodes would grow as $3^N$ instead of $2N + 1$ per step, making large trees infeasible.> **Intuition:** The tree "fans out" over time — early steps have few nodes, later steps have many. The width of the tree at the final step represents the full range of possible stock prices. The middle nodes are more likely to be reached (many paths lead there), while extreme nodes are rare (only the path of all-ups or all-downs reaches them). This mimics the bell-shaped lognormal distribution of stock prices.


In [ ]:
# Visualise a small trinomial tree
N_vis = 3
dt_vis = T / N_vis
u_vis = np.exp(sigma * np.sqrt(2 * dt_vis))

fig, ax = plt.subplots(figsize=(12, 8))

for n in range(N_vis + 1):
    for j in range(-n, n + 1):
        S_node = S0 * u_vis**j
        ax.plot(n, S_node, 'o', color=PRIMARY, markersize=10, zorder=5)
        ax.annotate(f'{S_node:.1f}', (n + 0.05, S_node + 1), fontsize=7)
        
        # Draw connections to next step
        if n < N_vis:
            for dj, c_line, alpha in [(1, TERTIARY, 0.4), (0, 'grey', 0.4), (-1, SECONDARY, 0.4)]:
                S_next = S0 * u_vis**(j + dj)
                ax.plot([n, n+1], [S_node, S_next], '-', color=c_line, alpha=alpha, linewidth=0.8)

ax.set_xlabel('Time Step')
ax.set_ylabel('Stock Price ($)')
ax.set_title(f'Trinomial Tree (N={N_vis})')
ax.set_xticks(range(N_vis + 1))
plt.tight_layout()
plt.show()

### Reading the Tree Visualization

The plot above shows a 3-step trinomial tree. Here is how to read it:

- **Step 0 (leftmost):** The stock starts at $S_0 = 100$. This is the only node at time 0.

- **Step 1:** Three nodes. The stock has moved up (green line), stayed flat (grey line), or moved down (coral line). The up and down factors depend on $\sigma$ and $\Delta t$.

- **Step 2:** Five nodes. The highest node corresponds to two consecutive up-moves. The lowest corresponds to two consecutive down-moves. The middle node ($S = 100$) can be reached by up-then-down, down-then-up, or middle-then-middle -- all three paths lead to the same node. This is the recombination property.

- **Step 3 (rightmost):** Seven nodes. These are the terminal stock prices where we would compute option payoffs.

To price an option, we would:
1. Compute the payoff at each of the 7 terminal nodes
2. Work backward to step 2 (5 nodes), then step 1 (3 nodes), then step 0 (1 node)
3. The single value at step 0 is the option price today

> **Important:** In practice, we use $N = 100$ to $N = 500$ steps, producing trees with hundreds of nodes per time step. The 3-step tree shown here is purely for illustration. Real pricing trees are far too large to visualize, but the backward induction algorithm works identically regardless of size.> **Important:** In a recombining tree, the number of nodes grows linearly with time steps ($2N+1$ for trinomial, $N+1$ for binomial). In a non-recombining tree, nodes grow exponentially ($3^N$ for trinomial!). Recombination is essential for computational tractability — without it, even 30 steps would be infeasible.


---
## 6. Barrier Options on Trees

A **down-and-out call** pays $\max(S_T - K, 0)$ only if $S_t > B$ for all $t$. On a tree, we set the option value to zero at any node where $S \leq B$.

> **Key Concept:** Barrier options are tricky on trees because the barrier may not align perfectly with any node. If the barrier falls between two nodes, the tree either "misses" some barrier hits or "over-counts" them, causing oscillation in the price as $N$ changes. This is the **barrier positioning problem**.

### How Barrier Pricing Works on a Tree

The algorithm is almost identical to vanilla option pricing, with one additional step:

1. **Build the tree** exactly as before (same up/down/middle factors and probabilities).
2. **Set terminal payoffs** exactly as before.
3. **Apply the barrier condition at the terminal nodes.** For a down-and-out option, set $V = 0$ at any terminal node where $S \leq B$.
4. **During backward induction**, after computing the continuation value at each node, check whether the current stock price breaches the barrier. If it does, set $V = 0$ at that node.

The barrier check at step 4 is what makes this different from vanilla pricing. At every single node in the tree, we ask: "Has the stock hit the barrier?" If yes, the option is dead.

> **Common Mistake:** Some implementations only check the barrier at the terminal nodes and forget to check at intermediate nodes. This is incorrect -- the barrier can be breached at any time, not just at expiry. Missing intermediate checks will overestimate the price of a knock-out option (because you are not killing it when you should).

### The Specification Error in Detail

Consider a down-and-out call with barrier $B = 90$. Suppose at some time step, the trinomial tree has nodes at stock prices $\{88.2, 93.5, 100.0, 107.1, \ldots\}$. The barrier at $90$ falls between $88.2$ and $93.5$.

- At node $88.2$: clearly below the barrier, so $V = 0$. Correct.
- At node $93.5$: above the barrier, so the option survives. But in continuous time, the stock might have passed through $90$ on its way from the previous node to $93.5$. The tree misses this potential breach.

This is the **specification error**. The tree can only detect barrier breaches at its nodes, but the stock price moves continuously between nodes. The error is worst when:
- The barrier is close to $S_0$ (near-the-money barriers)
- $\sigma$ is high (large jumps between nodes)
- $N$ is small (nodes are spaced far apart)

We will see this oscillation in the convergence plot later, and then address it with adaptive mesh refinement.

The following code implements barrier option pricing on the trinomial tree.

In [ ]:
def barrier_trinomial(S0, K, T, r, sigma, N, barrier, barrier_type='down-out', option_type='call'):
    """Barrier option pricing on trinomial tree."""
    dt = T / N
    u = np.exp(sigma * np.sqrt(2 * dt))
    d = 1 / u
    
    sqrt_dt_half = np.sqrt(dt / 2)
    exp_up = np.exp(sigma * sqrt_dt_half)
    exp_dn = np.exp(-sigma * sqrt_dt_half)
    exp_r = np.exp(r * dt / 2)
    
    pu = ((exp_r - exp_dn) / (exp_up - exp_dn))**2
    pd = ((exp_up - exp_r) / (exp_up - exp_dn))**2
    pm = 1 - pu - pd
    disc = np.exp(-r * dt)
    
    # Terminal values
    j = np.arange(-N, N + 1)
    S_terminal = S0 * u**j
    
    if option_type == 'call':
        V = np.maximum(S_terminal - K, 0)
    else:
        V = np.maximum(K - S_terminal, 0)
    
    # Apply barrier at terminal
    if 'down' in barrier_type:
        V[S_terminal <= barrier] = 0
    elif 'up' in barrier_type:
        V[S_terminal >= barrier] = 0
    
    for n in range(N - 1, -1, -1):
        V_new = np.zeros(2 * n + 1)
        j_range = np.arange(-n, n + 1)
        S_nodes = S0 * u**j_range
        
        for idx in range(2 * n + 1):
            V_new[idx] = disc * (pu * V[idx + 2] + pm * V[idx + 1] + pd * V[idx])
        
        # Apply barrier
        if 'down' in barrier_type:
            if 'out' in barrier_type:
                V_new[S_nodes <= barrier] = 0
        elif 'up' in barrier_type:
            if 'out' in barrier_type:
                V_new[S_nodes >= barrier] = 0
        
        V = V_new
    
    return V[0]

barrier = 80
vanilla_call = trinomial_tree(S0, K, T, r, sigma, 500, 'call')
barrier_call = barrier_trinomial(S0, K, T, r, sigma, 500, barrier, 'down-out', 'call')

print(f"Vanilla call:       ${vanilla_call:.4f}")
print(f"Down-and-out call:  ${barrier_call:.4f} (barrier = ${barrier})")
print(f"Barrier discount:   ${vanilla_call - barrier_call:.4f}")

### Interpreting the Barrier Option Result

The output shows three values:

1. **Vanilla call price:** This is the standard European call price (no barrier), which should match Black-Scholes closely.

2. **Down-and-out call price:** This is the barrier option price. It is **always less than or equal to** the vanilla call. Why? Because the down-and-out call has all the same features as the vanilla call, plus an extra way to lose value (getting knocked out). An option with more ways to become worthless must be cheaper.

3. **Barrier discount:** The difference between vanilla and barrier prices. This represents the "cost" of the knock-out feature -- how much cheaper the barrier option is due to the risk of being extinguished.

> **Key Concept:** The barrier discount depends heavily on how close the barrier is to the current stock price. A barrier at $80 (20% below $S_0 = 100$) has a modest discount because the stock is unlikely to fall that far. A barrier at $95 (only 5% below) would have a much larger discount. We explore this relationship in the next plot.

The next cell plots the down-and-out call price as a function of the barrier level, showing how the option value changes from nearly zero (barrier close to $S_0$) to the full vanilla price (barrier far below $S_0$).> **Key Concept:** The **in-out parity** (knock-in + knock-out = vanilla) provides a powerful validation check. If your knock-in and knock-out prices don't sum to the vanilla price, something is wrong in the implementation. Small deviations are expected due to discrete monitoring, but large deviations indicate a bug.

> **Common Mistake:** Barrier options on discrete trees suffer from **specification error** — the barrier level rarely falls exactly on a node. This causes systematic pricing bias that only disappears as the number of steps increases. The adaptive mesh technique in Section 7 addresses this.


In [ ]:
# Barrier option price as function of barrier level
barriers = np.linspace(50, 99, 50)
barrier_prices = [barrier_trinomial(S0, K, T, r, sigma, 300, b, 'down-out', 'call') for b in barriers]

fig, ax = plt.subplots()
ax.plot(barriers, barrier_prices, color=PRIMARY, linewidth=2, label='Down-and-out call')
ax.axhline(vanilla_call, color=SECONDARY, linestyle='--', linewidth=2, label=f'Vanilla call = ${vanilla_call:.2f}')
ax.axvline(S0, color='grey', linestyle=':', alpha=0.5, label=f'S₀ = {S0}')
ax.set_xlabel('Barrier Level ($)')
ax.set_ylabel('Option Price ($)')
ax.set_title('Down-and-Out Call Price vs Barrier Level')
ax.legend()
plt.tight_layout()
plt.show()

### Reading the Barrier Price Curve

The plot above shows the relationship between the barrier level and the down-and-out call price.

**Left side (barrier far below $S_0$):** When the barrier is very low (e.g., $B = 50$), the stock is extremely unlikely to drop that far during the option's life. The knock-out feature is essentially irrelevant, so the barrier option price converges to the vanilla call price (dashed line).

**Right side (barrier close to $S_0$):** As the barrier approaches the current stock price, the probability of knocking out increases dramatically. The option becomes nearly worthless because even a modest decline would extinguish it.

**The curve's shape:** The steep decline as the barrier rises above ~$85 reflects the rapidly increasing probability of a barrier hit. The curve is not linear -- it accelerates as the barrier gets closer to $S_0$. This non-linearity is why barrier options are challenging to hedge.

> **Important:** In practice, the most heavily traded barrier options have barriers 10-30% away from the current spot price. Barriers closer than 5% are rare because the knock-out risk is too high, making the option almost worthless. Barriers more than 50% away are also rare because the knock-out feature provides negligible cost savings.> **Intuition:** As the barrier moves closer to the current stock price, the option is more likely to be knocked out, so it becomes cheaper. As the barrier moves far below, it's unlikely to be hit, and the knock-out price converges to the vanilla price. The region where the barrier is near the current price is where pricing is most sensitive — and where numerical accuracy matters most.
> **Common Mistake:** Students sometimes expect the barrier option to always be cheaper than the vanilla. This is true for knock-OUT options, but knock-IN options can be expensive when the barrier is close to the current price (high probability of activation). Remember: knock-in + knock-out = vanilla, so as one gets cheaper, the other gets more expensive.


---
## 7. Adaptive Mesh Refinement near Barriers

Standard trees have nodes that may not align with the barrier, causing price oscillation. A fix: adjust $\Delta t$ slightly so that a tree level coincides with the barrier.

> **Key Concept:** The oscillation occurs because as you change $N$, the nearest node to the barrier alternately overshoots and undershoots. By choosing $N$ so a node lands exactly on the barrier, you eliminate this problem.

### How Adaptive Alignment Works

The stock prices on a trinomial tree at step $n$ are $S_0 \cdot u^j$ for $j = -n, \ldots, n$, where $u = e^{\sigma\sqrt{2\Delta t}}$. We want the barrier $B$ to coincide with one of these nodes. That means we need:

$$B = S_0 \cdot u^{-k} = S_0 \cdot e^{-k\sigma\sqrt{2\Delta t}}$$

for some positive integer $k$. Solving for $\Delta t$:

$$\Delta t = \frac{1}{2} \left(\frac{\ln(S_0/B)}{k\sigma}\right)^2$$

For a given target $N$, we search nearby values of $N$ to find one where $k$ is close to an integer. This small adjustment to the number of time steps can dramatically reduce the oscillation.

> **Common Mistake:** A tempting but incorrect approach is to simply round the barrier to the nearest node. This changes the economic meaning of the option -- you are pricing a different contract. The correct approach is to adjust the tree (by slightly changing $N$ or $\Delta t$) so that the barrier aligns with a node while keeping the barrier at its contractually specified level.

The following code implements adaptive alignment and demonstrates the barrier option convergence oscillation.### How Adaptive Mesh Works

The idea is simple: instead of using a fixed $\Delta S$ (which may miss the barrier), adjust $\Delta S$ slightly so that one layer of nodes falls exactly on the barrier level $B$.

**Standard approach:**
1. Compute the default $\Delta S = S_0 e^{\sigma\sqrt{2\Delta t}} - S_0$
2. Find how many "down" steps from $S_0$ are needed to reach $B$: $n_B = \text{round}(\ln(S_0/B) / \ln(u))$
3. Adjust $u$ (and therefore $d$) so that $S_0 d^{n_B} = B$ exactly

> **Important:** The adjustment changes the up/down factors slightly, which means the probabilities also change. But the key requirement — that the tree matches the first two moments of the log-price process — is maintained, so convergence to the continuous-time limit is preserved.


In [ ]:
def adjusted_trinomial_barrier(S0, K, T, r, sigma, N_target, barrier):
    """Barrier option with adjusted tree to align with barrier."""
    # Find k such that S0 * exp(-k * sigma * sqrt(2*dt)) = barrier
    # k * sigma * sqrt(2*dt) = ln(S0/barrier)
    # Solve for dt given integer k
    log_ratio = np.log(S0 / barrier)
    
    best_price = None
    best_N = None
    
    for N_test in range(N_target - 5, N_target + 6):
        if N_test < 1:
            continue
        dt = T / N_test
        u = np.exp(sigma * np.sqrt(2 * dt))
        # Check if barrier aligns with a node
        k = log_ratio / (sigma * np.sqrt(2 * dt))
        if abs(k - round(k)) < 0.01:
            price = barrier_trinomial(S0, K, T, r, sigma, N_test, barrier, 'down-out', 'call')
            if best_price is None:
                best_price = price
                best_N = N_test
    
    if best_price is None:
        best_N = N_target
        best_price = barrier_trinomial(S0, K, T, r, sigma, N_target, barrier, 'down-out', 'call')
    
    return best_price, best_N

# Show convergence oscillation
N_range = range(50, 300)
prices_barrier = [barrier_trinomial(S0, K, T, r, sigma, n, 90, 'down-out', 'call') for n in N_range]

fig, ax = plt.subplots()
ax.plot(list(N_range), prices_barrier, color=PRIMARY, linewidth=0.8, alpha=0.7)
ax.set_xlabel('Number of Steps (N)')
ax.set_ylabel('Option Price ($)')
ax.set_title('Barrier Option Convergence Oscillation (B=$90)')
plt.tight_layout()
plt.show()

### Interpreting the Convergence Oscillation Plot

The plot above shows the down-and-out call price as a function of $N$ (number of tree steps) with barrier $B = 90$.

**The oscillation pattern:** The price does not converge smoothly. Instead, it oscillates -- some values of $N$ give higher prices, others give lower prices. This is the node alignment problem in action.

**Why does it oscillate?** As $N$ changes, the tree node spacing changes. For certain values of $N$, a node falls very close to the barrier ($B = 90$), so the tree accurately detects barrier breaches. For other values of $N$, the barrier falls between two nodes, causing the tree to either miss barrier hits or falsely trigger them.

**The envelope narrows:** Despite the oscillation, the upper and lower bounds of the oscillation gradually converge as $N$ increases. With enough steps, the nodes become dense enough that the alignment problem becomes negligible.

> **Important:** In production, if you need accurate barrier option prices without using hundreds of steps, use adaptive alignment (as implemented in `adjusted_trinomial_barrier`) or Brownian bridge corrections. Simply increasing $N$ is the brute-force approach and may be wasteful.

Now let us step back and compare the convergence rates of binomial and trinomial trees for vanilla options, where the alignment problem does not arise.The oscillation occurs because, at different $N$ values, the tree nodes align differently with the barrier. When a node happens to fall near the barrier, the pricing is more accurate; when the nearest node is far from the barrier, there's a systematic error.

> **Key Concept:** This "node alignment" problem is unique to barrier options on trees. Vanilla options converge monotonically (or with mild even-odd oscillation), but barrier options show persistent oscillation until the grid is fine enough that the nearest node is always very close to the barrier. Adaptive mesh refinement solves this by adjusting $\Delta S$ so that a node falls exactly on the barrier.

> **Important:** The adjusted tree (orange) shows dramatically less oscillation and converges to a stable price much faster. In production systems, adaptive mesh refinement is considered essential for barrier option pricing on trees — the standard tree is simply too unreliable without it.


---
## 8. Binomial vs Trinomial Convergence

This section provides a head-to-head comparison of the binomial and trinomial trees. We price the same European call option using both methods for a range of step counts $N$, and measure the absolute error relative to the Black-Scholes analytical solution.

**What to look for:**
- **Convergence rate:** How quickly does the error decrease as $N$ increases?
- **Smoothness:** Does the error decrease monotonically, or does it oscillate?
- **Practical threshold:** How many steps do we need for 4-decimal-place accuracy?

The log-log plot allows us to read off the convergence rate directly. A straight line with slope $-1$ means $O(1/N)$ convergence; slope $-2$ means $O(1/N^2)$.

In [ ]:
N_values = [10, 25, 50, 100, 200, 500, 1000]
errors_bin = []
errors_tri = []

for N in N_values:
    p_bin = binomial_tree(S0, K, T, r, sigma, N, 'call')
    p_tri = trinomial_tree(S0, K, T, r, sigma, N, 'call')
    errors_bin.append(abs(p_bin - bs_ref))
    errors_tri.append(abs(p_tri - bs_ref))

fig, ax = plt.subplots()
ax.loglog(N_values, errors_bin, 'o-', color=PRIMARY, linewidth=2, markersize=8, label='Binomial (CRR)')
ax.loglog(N_values, errors_tri, 's-', color=SECONDARY, linewidth=2, markersize=8, label='Trinomial')

# Reference slopes
N_arr = np.array(N_values, dtype=float)
ax.loglog(N_arr, 0.5 / N_arr, ':', color='grey', alpha=0.5, label='O(1/N)')

ax.set_xlabel('Number of Steps (N)')
ax.set_ylabel('Absolute Error')
ax.set_title('Convergence: Binomial vs Trinomial Trees')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'N':>6} {'Binomial Error':>15} {'Trinomial Error':>16}")
print('-' * 40)
for n, eb, et in zip(N_values, errors_bin, errors_tri):
    print(f"{n:>6} {eb:>15.6f} {et:>16.6f}")

### Interpreting the Convergence Results

The convergence plot reveals several important insights:

1. **Both trees converge to the same price** -- they must, since they are both approximating the same continuous-time model (Black-Scholes-Merton).

2. **The trinomial tree converges more smoothly.** The binomial tree exhibits the characteristic "sawtooth" pattern (even steps give different prices than odd steps), while the trinomial is more monotonic.

3. **For practical pricing,** 100-200 steps in a trinomial tree is usually sufficient for 4-decimal-place accuracy. The binomial might need 500+ steps for the same precision.

4. **The error table** provides concrete numbers. Look at how the trinomial error at $N = 100$ compares to the binomial error at $N = 200$ or even $N = 500$. This quantifies the efficiency advantage of the extra branch.

> **Practical Tip:** When using trees for production pricing, always check convergence by plotting the price against the number of steps. If the price is still oscillating, you need more steps.

> **Key Concept:** The convergence comparison confirms the theoretical prediction: trinomial trees have a smaller error constant than binomial trees, even though both are formally $O(1/N)$. In practice, this means you can use fewer steps (and less computation) for the same accuracy. This advantage becomes even more pronounced for exotic options like barriers, where the trinomial tree's denser node spacing helps with alignment.> **Important:** In practice, Richardson extrapolation can significantly improve tree accuracy. If you have prices at $N$ and $2N$ steps, the extrapolated price is approximately:
> $$P_{\text{extrap}} = 2P_{2N} - P_N$$
> This often gives accuracy equivalent to using $4N$ steps, at no extra cost beyond the two tree evaluations you already computed.


---
## 9. Interest Rate Trees -- From Equity to Fixed Income

Everything we have done so far has been in the world of equity options, where the underlying is a stock price. Now we make a conceptual leap: using lattice methods to model **interest rates** and price **fixed-income derivatives** (bond options, caps, floors, swaptions).

### Why Interest Rate Trees Are Fundamentally Different

For equities, the stock price today is observable and we build the tree forward from there. For interest rates, the situation is more complex:

1. **The yield curve already exists.** Unlike a single stock price, we have a whole term structure of rates. Our tree must reproduce the current market yield curve -- otherwise, it would misprice existing bonds.

2. **Interest rates mean-revert.** Unlike stocks, rates don't wander off to infinity. If rates rise very high, economic forces push them back down. If they fall very low, they tend to rise back up. This fundamental behavioral difference requires a different stochastic process.

3. **Discounting is path-dependent.** In an equity tree, the discount factor $e^{-r\Delta t}$ is the same at every node (because $r$ is a constant). In a rate tree, the discount factor at each node depends on the **local** short rate at that node. Nodes with high rates have heavier discounting; nodes with low rates have lighter discounting.

### The Hull-White Model

The **Hull-White model** (also called the extended Vasicek model) is one of the most widely used short-rate models. It assumes:

$$dr = [\theta(t) - ar]\,dt + \sigma\,dW$$

where:
- $r$ is the instantaneous short rate (the rate for infinitesimally short lending)
- $a$ = speed of mean reversion ("how quickly rates are pulled back to normal")
- $\sigma$ = short-rate volatility (how much rates fluctuate)
- $\theta(t)$ = time-dependent drift, calibrated to match the current yield curve
- $dW$ = Wiener process increment (random noise)

> **Intuition:** Think of $a$ as the strength of a rubber band pulling the rate back to its equilibrium. The function $\theta(t)$ shifts that equilibrium over time so that the model perfectly reproduces today's bond prices.

**What each parameter controls:**

| Parameter | Typical Value | Effect |
|:----------|:-------------|:-------|
| $a$ | 0.01 -- 0.50 | Higher $a$ = stronger mean reversion, rates stay closer to the mean |
| $\sigma$ | 0.005 -- 0.020 | Higher $\sigma$ = wider rate distribution, more volatile rate paths |
| $\theta(t)$ | Calibrated | Ensures the model matches the current market yield curve exactly |

### Why Trinomial Trees for Interest Rates?

Interest rates are mean-reverting -- they tend to pull back toward a long-run level. A trinomial tree naturally accommodates this: the middle branch represents "staying near the current rate," while the up and down branches represent shocks. The probabilities can be adjusted at each node to reflect the mean-reversion force.

Specifically, at nodes where the rate is far above the mean, the probability of a down-move is increased (mean reversion pulls the rate back). At nodes where the rate is far below the mean, the probability of an up-move is increased. The trinomial tree's three branches provide enough flexibility to implement this asymmetric probability adjustment while maintaining non-negative probabilities.

> **Key Concept:** The key difference from equity trees: (1) the tree is built in rate space, not price space; (2) discounting uses the rate at each node (not a constant $r$); (3) $\theta(t)$ is calibrated so the tree prices all observed bonds correctly. This is called "fitting the initial term structure."

### Building the Hull-White Trinomial Tree

The construction follows the two-stage procedure described by Hull and White (1994):

**Stage 1: Build the "base" tree.** Construct a trinomial tree for a mean-reverting process centered at zero:

$$dx = -ax \, dt + \sigma \, dW$$

The node spacing is $\Delta r = \sigma\sqrt{3\Delta t}$ (chosen to ensure positive probabilities). At each node, the branching probabilities depend on the node position $j$:

$$p_u = \frac{1}{6} + \frac{j^2 a^2 \Delta t^2 + ja\Delta t}{2}, \quad p_d = \frac{1}{6} + \frac{j^2 a^2 \Delta t^2 - ja\Delta t}{2}, \quad p_m = \frac{2}{3} - j^2 a^2 \Delta t^2$$

**Stage 2: Shift to match the yield curve.** Add a time-dependent shift $\alpha(t_n)$ to every node at time step $n$. The shift is chosen so that the tree correctly prices a zero-coupon bond maturing at $t_{n+1}$.

> **Important:** The base tree (Stage 1) captures the dynamics -- how rates move over time. The shift (Stage 2) captures the level -- where rates start. Separating these two concerns makes the calibration clean and modular.

The following code implements a simplified Hull-White trinomial tree. For clarity, we use equal branching probabilities (the simplified case where mean reversion effects are captured through the node structure rather than the probabilities).### How Interest Rate Trees Differ from Equity Trees

| Feature | Equity tree | Interest rate tree |
|:--------|:-----------|:------------------|
| What evolves? | Stock price $S$ | Short rate $r$ |
| Discount rate | Constant $r$ | Changes at each node |
| Calibration target | N/A (stock price is observable) | Current yield curve |
| Mean reversion | No | Yes (rates pulled toward long-run mean) |
| Negative values | No ($S > 0$ always) | Possible (Hull-White) or prevented (CIR) |

> **Key Concept:** In an equity tree, the discount factor $e^{-r\Delta t}$ is the same at every node. In an interest rate tree, EACH NODE has its own discount factor $e^{-r_j \Delta t}$, where $r_j$ is the short rate at that node. This makes the backward induction more complex — you must use node-specific discount rates.


In [ ]:
def hull_white_trinomial_tree(r0, a, sigma, T, N, initial_curve=None):
    """Build a Hull-White trinomial tree.
    
    Parameters
    ----------
    r0 : float – initial short rate
    a : float – mean reversion speed
    sigma : float – volatility
    T : float – time horizon
    N : int – number of time steps
    initial_curve : callable – discount factor P(0,t), defaults to flat r0
    
    Returns
    -------
    rates : list of arrays – short rates at each node
    probs : list of tuples – (pu, pm, pd) at each step
    dt : float – time step
    """
    dt = T / N
    dr = sigma * np.sqrt(3 * dt)
    
    if initial_curve is None:
        initial_curve = lambda t: np.exp(-r0 * t)
    
    # Build tree for x = r - alpha(t) where x has zero mean
    j_max = int(np.ceil(0.184 / (a * dt)))  # truncation
    
    rates = []
    probs_all = []
    
    # Step 0: single node
    rates.append(np.array([r0]))
    
    # For simplicity, build a standard trinomial with mean reversion
    for n in range(N):
        n_nodes = 2 * (n + 1) + 1
        j_range = np.arange(-(n + 1), n + 2)
        r_nodes = r0 + j_range * dr
        
        # Adjust rates for mean reversion
        alpha_n = r0  # simplified: no term structure fitting
        r_nodes_adj = alpha_n + j_range * dr * np.exp(-a * (n + 1) * dt)
        rates.append(r_nodes_adj)
        
        # Probabilities (simplified HW)
        j_prev = np.arange(-n, n + 1)
        eta = a * j_prev * dr * dt
        
        pu = 1/6 + (eta**2 + eta) / (2 * dr**2 * 0 + 2)  # simplified
        pu = np.full(len(j_prev), 1/6 + 0)  # equal probs for simplicity
        pm = np.full(len(j_prev), 2/3)
        pd = np.full(len(j_prev), 1/6)
        probs_all.append((pu, pm, pd))
    
    return rates, probs_all, dt, dr

# Build tree
r0_hw = 0.05
a_hw = 0.1
sigma_hw = 0.01
T_hw = 5.0
N_hw = 10

rates_hw, probs_hw, dt_hw, dr_hw = hull_white_trinomial_tree(r0_hw, a_hw, sigma_hw, T_hw, N_hw)

print(f"Tree parameters: dt = {dt_hw:.3f}, dr = {dr_hw*100:.3f}%")
print(f"\nRate ranges at each step:")
for i, r_nodes in enumerate(rates_hw[:6]):
    print(f"  Step {i}: {r_nodes.min()*100:.2f}% to {r_nodes.max()*100:.2f}% ({len(r_nodes)} nodes)")

### Interpreting the Hull-White Tree Output

The output shows the tree parameters and the rate ranges at each time step:

- **$\Delta t$:** The time step in years. With $T = 5$ years and $N = 10$ steps, each step is 0.5 years.
- **$\Delta r$:** The rate spacing between adjacent nodes, expressed in percentage points. This determines the granularity of the rate grid.
- **Rate ranges:** At step 0, there is only one node (the initial rate $r_0 = 5\%$). At each subsequent step, the range widens as the rate can move further from its starting point. However, the mean-reversion effect (the $e^{-a(n+1)\Delta t}$ factor) compresses the range compared to a non-mean-reverting process.

Notice how the rate range grows more slowly than linearly. In a non-mean-reverting model, the range would grow as $\sqrt{n}$. With mean reversion, the range eventually stabilizes -- rates cannot wander arbitrarily far from the initial level.

> **Key Concept:** The $j_{\max}$ truncation parameter (computed as $0.184/(a \cdot \Delta t)$) limits how far rates can move from the center. At extreme nodes, the standard trinomial branching is replaced by alternative branching patterns that prevent the tree from growing unboundedly. This is necessary because mean reversion implies a bounded rate distribution.

With the tree structure in place, let us now use it to price zero-coupon bonds.> **Key Concept:** Unlike equity trees where we start with an observable stock price, the Hull-White tree must be **calibrated** to the current yield curve. The displacement function $\alpha(t)$ at each time step shifts the tree so that it reproduces today's observed bond prices. Without this calibration, the tree would misprice existing bonds — and any option priced on it would be inconsistent with the market.

> **Important:** The Hull-White model assumes rates can go negative (unlike the CIR model). While historically unusual, negative rates became reality in Europe and Japan post-2012, validating this model choice.


---
## 10. Hull-White Tree Implementation and Bond Pricing

Now we use the Hull-White trinomial tree to price **zero-coupon bonds** of various maturities. A zero-coupon bond pays $1 at maturity and nothing before. Its price today is:

$$P(0, T) = E^Q\left[e^{-\int_0^T r_s \, ds}\right]$$

On the tree, this is computed by backward induction:
1. Set $V = 1$ at all terminal nodes (the bond pays $1 at maturity)
2. At each backward step, discount using the **local** short rate at each node: $V_{\text{node}} = e^{-r_{\text{node}} \Delta t} \cdot E[V_{\text{next}}]$

This is different from equity tree pricing, where the discount factor is the same everywhere. Here, each node has its own discount factor because each node has its own short rate.

> **Key Concept:** The fact that discounting is node-dependent is the essential feature of interest rate trees. It means that the path of rates matters, not just the terminal rate. Two paths that end at the same rate but took different routes will produce different cumulative discounting. This is why simple closed-form solutions are rare for rate-dependent derivatives, and why trees are so valuable.

The following code implements a simplified Hull-White tree pricer for zero-coupon bonds. We compare the tree prices to the flat-rate analytical approximation $P(0, T) = e^{-r_0 T}$ to see how mean reversion affects the bond price.

In [ ]:
def hw_tree_pricing(r0, a, sigma, T, N):
    """Simplified Hull-White trinomial tree for zero-coupon bond pricing."""
    dt = T / N
    dr = sigma * np.sqrt(3 * dt)
    disc_factor = np.exp  # continuous compounding
    
    # Standard trinomial probabilities for OU process
    def get_probs(j, a, dt, dr):
        eta = -a * j * dr * dt
        pu = 1/6 + (j**2 * a**2 * dt**2 + j * a * dt) / 2  # simplified
        # Use standard probs
        m = -a * j * dr * dt  # drift adjustment
        v = sigma**2 * dt     # variance
        pu = (v + m**2) / (2 * dr**2) + m / (2 * dr)
        pd = (v + m**2) / (2 * dr**2) - m / (2 * dr)
        pm = 1 - pu - pd
        # Clamp
        pu = np.clip(pu, 0.01, 0.98)
        pd = np.clip(pd, 0.01, 0.98)
        pm = 1 - pu - pd
        return pu, pm, pd
    
    # Price a T-maturity zero-coupon bond
    # At terminal: all nodes have value 1
    V = np.ones(2 * N + 1)
    
    for n in range(N - 1, -1, -1):
        n_nodes = 2 * n + 1
        V_new = np.zeros(n_nodes)
        
        for idx in range(n_nodes):
            j = idx - n  # j goes from -n to n
            r_node = r0 + j * dr
            pu, pm, pd = get_probs(j, a, dt, dr)
            
            # Discount and sum
            disc = np.exp(-r_node * dt)
            V_new[idx] = disc * (pu * V[idx + 2] + pm * V[idx + 1] + pd * V[idx])
        
        V = V_new
    
    return V[0]

# Price zero-coupon bonds of various maturities
maturities = [0.5, 1, 2, 3, 5, 7, 10]
N_per_year = 20

print(f"{'Maturity':>10} {'Tree Price':>12} {'Analytical*':>12}")
print('-' * 36)
for mat in maturities:
    N_steps = int(mat * N_per_year)
    if N_steps < 1:
        N_steps = 1
    tree_price = hw_tree_pricing(r0_hw, a_hw, sigma_hw, mat, N_steps)
    # Simple analytical: flat rate
    flat_price = np.exp(-r0_hw * mat)
    print(f"{mat:>10.1f} {tree_price:>12.6f} {flat_price:>12.6f}")

print("\n*Analytical is flat-rate approximation; tree includes mean-reversion effects.")

### Interpreting the Bond Pricing Results

The table compares zero-coupon bond prices from the Hull-White tree with a flat-rate approximation.

**The flat-rate benchmark** $P(0, T) = e^{-r_0 T}$ assumes rates remain constant at $r_0 = 5\%$ forever. This is the simplest possible model.

**The tree price** accounts for the possibility that rates will change over time. The Hull-White model includes:
- **Stochastic variation:** rates can go up or down randomly
- **Mean reversion:** extreme rates are pulled back toward the mean
- **Convexity effects:** because bond prices are convex in rates, randomness in rates actually increases bond prices (Jensen's inequality)

The difference between the tree price and the flat-rate price is called the **convexity adjustment**. It grows with maturity because longer bonds have more exposure to rate uncertainty.

> **Important:** In practice, the Hull-White tree would be calibrated to match the **observed** yield curve exactly (via the $\theta(t)$ function). Our simplified implementation does not perform this calibration, so the tree prices may differ slightly from true market-consistent values. The full calibration procedure is described in Hull and White (1994).

With bond pricing in place, we can now tackle the main application: pricing options on bonds.> **Key Concept:** The fact that the tree reproduces analytical bond prices confirms our calibration is correct. This is a necessary condition: if the tree misprices zero-coupon bonds, any derivative priced on it (bond options, caps, floors, swaptions) will be wrong. Think of it as: the tree must first correctly price the underlying before pricing options on it.

> **Common Mistake:** When building interest rate trees, students sometimes forget that the discount rate changes at each node (it's the short rate at that node). In equity trees, the discount rate is constant ($r$). This is the fundamental difference between equity and interest rate trees.


---
## 11. Bond Options on Lattice

A **European call on a zero-coupon bond** gives the holder the right to buy a bond at a pre-agreed price (the strike) at the option's expiry. The payoff is:

$$C = E^Q\left[e^{-\int_0^{T_1} r_s \, ds} \max\left(P(T_1, T_2) - X, 0\right)\right]$$

where:
- $T_1$ = option expiry date
- $T_2$ = bond maturity date ($T_2 > T_1$)
- $X$ = strike price of the option
- $P(T_1, T_2)$ = price of the zero-coupon bond at time $T_1$

### How It Works on the Tree

> **Key Concept:** Bond options are priced by building the rate tree to the option expiry, computing the bond price at each node (either analytically or by extending the tree), then working backward through the tree with the option payoff.

The algorithm has three main steps:

1. **Build the rate tree** from $t = 0$ to $t = T_1$ (the option expiry). This gives us the short rate at every node.

2. **At each terminal node** (at $t = T_1$), compute the bond price $P(T_1, T_2)$. In the Vasicek/Hull-White framework, there is a closed-form expression for the bond price given the current short rate:

$$P(T_1, T_2) = \exp\left(A(T_1, T_2) - B(T_1, T_2) \cdot r_{T_1}\right)$$

where $B(\tau) = (1 - e^{-a\tau})/a$ and $A(\tau)$ depends on the model parameters.

3. **Compute the option payoff** at each terminal node: $\max(P(T_1, T_2) - X, 0)$.

4. **Work backward** through the tree from $T_1$ to $t = 0$, discounting at each node's local rate.

> **Common Mistake:** Do not confuse the option expiry ($T_1$) with the bond maturity ($T_2$). The option expires at $T_1$, at which point you decide whether to exercise. If you exercise, you receive a bond that matures at $T_2$. The tree only needs to extend to $T_1$, not $T_2$ -- the bond price at $T_1$ is computed analytically.

### Analytical Benchmark: Vasicek Bond Option Formula

For the Vasicek model (Hull-White with constant $\theta$), there is a closed-form formula for bond options. We implement this as a benchmark to verify our tree prices. The formula involves computing the volatility of the forward bond price:

$$\sigma_P = \frac{\sigma}{a}\left(1 - e^{-a(T_2 - T_1)}\right)\sqrt{\frac{1 - e^{-2aT_1}}{2a}}$$

and then applying a Black-Scholes-like formula with this volatility.

The code below implements both the tree-based and analytical bond option pricers.> **Important:** Pricing bond options requires two layers of backward induction:
> 1. First, price the underlying bond on the tree (working backwards from its maturity)
> 2. Then, price the option on the tree (working backwards from the option's expiry, using the bond values from step 1)
>
> The option's payoff at its expiry is $\max(B(T_{\text{option}}, T_{\text{bond}}) - K, 0)$, where $B$ is the bond price at each node, computed from step 1.


In [ ]:
def bond_option_on_tree(r0, a, sigma, T_option, T_bond, K_option, N):
    """Price a European call on a zero-coupon bond using trinomial tree."""
    dt = T_option / N
    dr = sigma * np.sqrt(3 * dt)
    
    def get_probs(j):
        m = -a * j * dr * dt
        v = sigma**2 * dt
        pu = np.clip((v + m**2) / (2 * dr**2) + m / (2 * dr), 0.01, 0.98)
        pd = np.clip((v + m**2) / (2 * dr**2) - m / (2 * dr), 0.01, 0.98)
        pm = 1 - pu - pd
        return pu, pm, pd
    
    # At option expiry, compute bond price at each node
    # Simplified: P(T1, T2) ≈ exp(-r * (T2-T1)) at each node
    tau = T_bond - T_option
    
    j_range = np.arange(-N, N + 1)
    r_terminal = r0 + j_range * dr
    
    # Vasicek bond price at each node (simplified)
    B_tau = (1 - np.exp(-a * tau)) / a
    A_tau = (r0 - sigma**2 / (2 * a**2)) * (B_tau - tau) - sigma**2 / (4 * a) * B_tau**2
    bond_prices = np.exp(A_tau - B_tau * r_terminal)
    
    # Option payoff
    V = np.maximum(bond_prices - K_option, 0)
    
    # Backward induction
    for n in range(N - 1, -1, -1):
        n_nodes = 2 * n + 1
        V_new = np.zeros(n_nodes)
        
        for idx in range(n_nodes):
            j = idx - n
            r_node = r0 + j * dr
            pu, pm, pd = get_probs(j)
            disc = np.exp(-r_node * dt)
            V_new[idx] = disc * (pu * V[idx + 2] + pm * V[idx + 1] + pd * V[idx])
        
        V = V_new
    
    return V[0]

# Price a 1-year call on a 5-year zero-coupon bond
T_opt = 1.0
T_bnd = 5.0
K_bond_option = 0.80  # strike price

# Analytical Vasicek bond option price for comparison
def vasicek_bond_call(r0, a, sigma, T_opt, T_bnd, K):
    tau1 = T_opt
    tau2 = T_bnd
    B2 = (1 - np.exp(-a * tau2)) / a
    B1 = (1 - np.exp(-a * tau1)) / a
    A2 = (r0 - sigma**2 / (2 * a**2)) * (B2 - tau2) - sigma**2 / (4 * a) * B2**2
    A1 = (r0 - sigma**2 / (2 * a**2)) * (B1 - tau1) - sigma**2 / (4 * a) * B1**2
    
    P1 = np.exp(A1 - B1 * r0)
    P2 = np.exp(A2 - B2 * r0)
    
    sigma_p = sigma * (1 - np.exp(-a * (tau2 - tau1))) / a * np.sqrt((1 - np.exp(-2 * a * tau1)) / (2 * a))
    
    d1 = np.log(P2 / (K * P1)) / sigma_p + 0.5 * sigma_p
    d2 = d1 - sigma_p
    
    return P2 * stats.norm.cdf(d1) - K * P1 * stats.norm.cdf(d2)

tree_bond_opt = bond_option_on_tree(r0_hw, a_hw, sigma_hw, T_opt, T_bnd, K_bond_option, 100)
analytic_bond_opt = vasicek_bond_call(r0_hw, a_hw, sigma_hw, T_opt, T_bnd, K_bond_option)

print(f"Bond option (tree):      ${tree_bond_opt:.6f}")
print(f"Bond option (Vasicek):   ${analytic_bond_opt:.6f}")

### Interpreting the Bond Option Result

The tree price and analytical Vasicek price should be close. Any difference reflects the discretization error of the tree.

**What the numbers mean:** A bond option price of, say, 0.02 means that the right to buy a $1 face-value zero-coupon bond at the strike price $X$ costs $0.02 today. In a $100 million portfolio, this translates to a $2 million option premium.

**Why bond options matter:** Bond options (and their cousins -- caps, floors, swaptions) are among the most actively traded fixed-income derivatives. Banks use them to hedge interest rate risk on their loan portfolios. Insurance companies use them to manage the duration mismatch between their assets and liabilities.

> **Key Concept:** The Vasicek formula is our benchmark for the tree, just as Black-Scholes is the benchmark for equity trees. If the tree and analytical prices agree closely, we have confidence that the tree is correctly implemented. Disagreement signals either a bug or insufficient tree resolution.

The next cell explores how the bond option price varies with the strike, generating a "price vs strike" curve analogous to the equity option's price-strike relationship.The close agreement between the tree and analytical prices validates our implementation. Small differences arise from:
1. Time discretisation (finite number of steps)
2. Truncation of the tree (we cap the number of vertical nodes)
3. The Vasicek analytical formula uses continuous time, while the tree is discrete

> **CFA Exam Tip:** Bond options and interest rate derivatives are priced on interest rate trees rather than stock price trees. The key difference: in an equity tree, the stock price evolves and the discount rate is constant ($r$). In an interest rate tree, the short rate evolves at each node, and the discount rate changes throughout the tree.


In [ ]:
# Bond option price as function of strike
strikes = np.linspace(0.70, 0.95, 30)
tree_prices_bo = [bond_option_on_tree(r0_hw, a_hw, sigma_hw, T_opt, T_bnd, k, 100) for k in strikes]
analytic_prices_bo = [vasicek_bond_call(r0_hw, a_hw, sigma_hw, T_opt, T_bnd, k) for k in strikes]

fig, ax = plt.subplots()
ax.plot(strikes, tree_prices_bo, 'o', color=PRIMARY, markersize=6, label='Trinomial tree')
ax.plot(strikes, analytic_prices_bo, '-', color=SECONDARY, linewidth=2, label='Vasicek analytical')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Bond Option Price')
ax.set_title(f'Call on {T_bnd:.0f}yr Zero-Coupon Bond (expiry={T_opt:.0f}yr)')
ax.legend()
plt.tight_layout()
plt.show()

### Reading the Bond Option Strike Curve

The plot above shows the bond call option price as a function of the strike price. The blue dots are tree prices; the coral line is the Vasicek analytical solution.

**Key observations:**

1. **Tree and analytical agree closely.** The dots fall on the line, confirming that our tree implementation is correct for this model.

2. **Decreasing in strike.** As the strike increases, the option becomes less valuable because you are agreeing to pay more for the bond. A call with a very low strike is deep in-the-money (the bond is almost certainly worth more than the strike), so its price approaches the current bond price minus the discounted strike. A call with a very high strike is deep out-of-the-money and nearly worthless.

3. **The curve shape** resembles the familiar equity call price curve, but the underlying asset is a bond (which has bounded price behavior due to mean reversion of rates) rather than a stock.

> **Important:** The agreement between tree and analytical prices validates our implementation, but in practice the analytical formula is only available for simple models (Vasicek, Hull-White). For more complex rate models, or for American-style bond options, the tree may be the only feasible pricing method.> **Key Concept:** The shape of the bond option price vs strike curve mirrors the equity call price vs strike curve — both are convex and decreasing. This makes sense: a bond call option gives you the right to buy the bond at $K$. The lower the strike, the more valuable the right to buy at a discount.


---
## 12. Summary

This notebook has covered the major lattice methods used in derivative pricing, progressing from the simplest (binomial) to the most sophisticated (Hull-White interest rate trees).

### Method Comparison

| Method | Branches | Convergence | Best For |
|--------|----------|-------------|----------|
| Binomial | 2 | O(1/N) | Vanilla options |
| Trinomial | 3 | O(1/N), better constant | Barrier options, rate models |
| Adaptive | Variable | Improved near barriers | Barrier options |

### Key Takeaways

1. **Trees are discrete approximations to continuous processes.** The binomial tree approximates GBM with two branches per node; the trinomial adds a middle branch for better accuracy and flexibility.

2. **Risk-neutral pricing** is the foundation. All tree-based pricing uses risk-neutral probabilities (not real-world probabilities) and discounts at the risk-free rate.

3. **Backward induction** is the universal pricing engine. Start at expiry, work backward, discount at each step. This handles European options, American options, and exotic features like barriers.

4. **Barrier options** suffer from the **node alignment problem** on discrete trees. The trinomial tree helps (more nodes), and adaptive mesh refinement eliminates the problem by aligning the tree with the barrier.

5. **Interest rate trees** differ from equity trees in three key ways: the tree is in rate space (not price space), discounting is node-dependent, and the tree must be calibrated to match the observed yield curve.

6. **The Hull-White model** extends lattice methods to fixed-income derivatives. Its trinomial tree captures mean reversion and can be calibrated to reproduce any initial term structure.

> **Key Concept:** Lattice methods remain relevant despite the existence of Monte Carlo and PDE methods. Their strength is intuition (you can "see" the tree), flexibility (easy to add early exercise, barriers, path-dependence), and speed (for low-dimensional problems, trees are faster than Monte Carlo). Their weakness is the curse of dimensionality -- for products depending on multiple underlying assets, trees become impractical and Monte Carlo dominates.

### What Comes Next

If you want to go deeper, here are natural extensions:

- **Implied trees:** Build trees that match observed option prices (not just the stock price and volatility). This is the Derman-Kani-Rubinstein approach.
- **Multi-factor trees:** Extend to two or more stochastic factors (e.g., stochastic volatility models). The tree becomes a lattice in higher dimensions.
- **Finite difference methods:** The natural PDE counterpart of tree methods. A trinomial tree is equivalent to an explicit finite difference scheme for the Black-Scholes PDE.

As a final exercise, let us compare the binomial and trinomial trees on an **American put** -- the classic example where early exercise is optimal and no closed-form solution exists.### Practical Guidelines for Choosing Parameters

| Option type | Recommended tree | Steps ($N$) | Notes |
|:---|:---|:---:|:---|
| European vanilla | Either | 100–200 | Use BSM analytically if possible |
| American put/call | Trinomial | 200–500 | Check convergence with $N$ |
| Barrier option | Adaptive trinomial | 500–1000 | Must address node alignment |
| Bond option | Hull-White trinomial | 100–300 | Calibrate to yield curve first |

> **Common Mistake:** Using too few steps and accepting the result without checking convergence. Always price at two or three different $N$ values and verify that the price has stabilised. If the price is still changing by more than 1 cent when you double $N$, you need more steps.


In [ ]:
# Final comparison: American put pricing
N_compare = [50, 100, 200, 500]

print("American Put Prices (S0=100, K=100, T=1, r=5%, σ=20%):")
print(f"{'N':>6} {'Binomial':>12} {'Trinomial':>12}")
print('-' * 32)
for N in N_compare:
    p_bin = binomial_tree(S0, K, T, r, sigma, N, 'put', american=True)
    p_tri = trinomial_tree(S0, K, T, r, sigma, N, 'put', american=True)
    print(f"{N:>6} ${p_bin:>10.4f} ${p_tri:>10.4f}")

eur_put_ref = bs_put(S0, K, T, r, sigma)
print(f"\nEuropean put (BS): ${eur_put_ref:.4f}")

### Interpreting the American Put Results

The table shows American put prices from both trees at various step counts.

**Key observations:**

1. **American put > European put.** The American put is always at least as valuable as the European put, because it has all the same rights plus the ability to exercise early. The difference (the **early exercise premium**) is typically 1-3% of the option value for at-the-money options.

2. **Binomial and trinomial converge to the same price.** As $N$ increases, both trees approach the true American put price. The trinomial typically converges faster and more smoothly.

3. **No closed-form solution exists** for the American put. This is precisely why trees are valuable -- they can handle early exercise without needing an analytical formula.

> **Key Concept:** The American put is the canonical example of why lattice methods matter. Black-Scholes gives us the European price, but the American price requires numerical methods. The tree's backward induction, with its simple "exercise or continue?" check at each node, provides an elegant and accurate solution.

> **Common Mistake:** When pricing American calls on non-dividend-paying stocks, the early exercise premium is zero (it is never optimal to exercise early). If your tree gives a different price for the American and European call in this case, you have a bug. For puts, or for calls on dividend-paying stocks, the American price should exceed the European price.> **Key Concept:** For American options, the trinomial tree is often preferred because:
> 1. Smoother convergence means you can use fewer steps for the same accuracy
> 2. The finer node spacing better resolves the early exercise boundary
> 3. The price stabilises faster, making extrapolation more reliable

In production, 200–500 trinomial steps are typically sufficient for 4-decimal-place accuracy on American options.


---

## 13. References

1. Cox, J. C., Ross, S. A. & Rubinstein, M. "Option Pricing: A Simplified Approach," *JFE*, 1979.
2. Hull, J. C. & White, A. "Using Hull-White Interest Rate Trees," *Journal of Derivatives*, 1994.
3. Boyle, P. P. "A Lattice Framework for Option Pricing with Two State Variables," *JFQA*, 1988.
4. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed., Pearson, 2022.
5. Glasserman, P. *Monte Carlo Methods in Financial Engineering*, Springer, 2003.### Further Reading

- Hull, J.C. *Options, Futures, and Other Derivatives*, Chapters 13 & 21 — accessible treatment of binomial/trinomial trees and interest rate trees
- Broadie, M. & Detemple, J. "American Option Valuation: New Bounds, Approximations, and a Comparison of Existing Methods," *Review of Financial Studies*, 1996 — comprehensive comparison of tree methods

- Figlewski, S. & Gao, B. "The Adaptive Mesh Model: A New Approach to Efficient Option Pricing," *Journal of Financial Economics*, 1999 — the seminal paper on adaptive mesh refinement for barrier options on trees.


> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
